<a href="https://colab.research.google.com/github/T-Chiunzi/Tava-Smart-Finance-Assistant/blob/main/Savings_Assistant_Tester.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **The following shows my code building processes without the use of the hands-on AI package.**

# **App Theme: Savings Buddy**

Intital Idea Pseudocode:

Send greetings to the user

Display Dashboard

Ask the user to input their salary

Ask user what they want to save for and how much of their salary they want to dedicate to it

User inputs their savings goals or 0 to stop

Display to the user that they should input their expenses

Ask the user if they want to input their expenses via CSV or manually.

If via CSV, run it

If manually, ask the user to input expenses or 0 to stop

Read the data

Output table of expenses

Output a pie chart of expenses

If expenses are more than what they want to save, then advise which expenses they should reduce

Ask the user for how long they want to save for their savings goal by months or indefinitely (for emergency savings)

Ask the user if they want to increase the amount they save by (by % or by money)

Display a timeline of savings with visual of a person running along the line


In [1]:
!pip install gradio

In [2]:
!pip install gradio pandas hands-on-ai
import os

os.environ['HANDS_ON_AI_SERVER'] = 'https://ollama.serveur.au'
os.environ['HANDS_ON_AI_MODEL'] = 'llama3.2'
os.environ['HANDS_ON_AI_API_KEY'] = 'isys2001-assignment-key'

print("✅ 🔑 Hands-on-AI configured successfully!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 1.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 178.2/178.2 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 33.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.0/253.0 kB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 360.7/360.7 kB 18.3 MB/s eta 0:00:00
  Attempting uninstall: jiter
    Found existing installation: jiter 0.14.0
    Uninstalling jiter-0.14.0:
      Successfully uninstalled jiter-0.14.0
✅ 🔑 Hands-on-AI configured successfully!


In [3]:
from hands_on_ai.chat import get_response

try:
    # Testing the precise function verified in your class handout
    test_response = get_response("Hello! This is a connection handshake test.")
    print("🚀 Hands-on-AI connection successful!")
    print(f"Response from campus server: {test_response}")
except Exception as e:
    print(f"❌ Connection issue still persisting: {e}")

⚙️ Switching to model: llama3.2 ... (may take a few seconds)


[WARNING] Error during request (attempt 1): Connection error.


Hang tight, I'm thinking... trying again!


[WARNING] Error during request (attempt 2): Connection error.


🚀 Hands-on-AI connection successful!
Response from campus server: ❌ Error: Connection error.


In [ ]:
import gradio as gr
import pandas as pd
from datetime import datetime
import calendar

# =====================================================================
# SYSTEM BACKEND ANALYTICS ENGINE (Savings Buddy Pipeline)
# =====================================================================

def calculate_savings_metrics(ledger, salary, emergency_amt, has_secondary, sec_cost, sec_date_str):
    """
    Calculates the financial dynamics for the savings dashboard.
    Tracks spend, target monthly rates for goals, and net surplus.
    """
    # 1. Total Expenses Calculation from standard ledger inputs
    total_spent = sum(tx["Amount"] for tx in ledger) if ledger else 0.0
    disposable_before_savings = max(0.0, salary - total_spent)

    # 2. Parse Secondary Goal Targets
    secondary_monthly_target = 0.0
    days_to_secondary = 0
    months_to_secondary = 1

    if has_secondary and sec_cost and sec_cost > 0 and sec_date_str:
        try:
            target_date = datetime.strptime(sec_date_str.strip(), "%Y-%m-%d").date()
            today = datetime.now().date()
            if target_date > today:
                # Calculate approximate remaining months
                days_to_secondary = (target_date - today).days
                months_to_secondary = max(1, round(days_to_secondary / 30.41))
                secondary_monthly_target = sec_cost / months_to_secondary
        except ValueError:
            pass # Gracefully handle malformed dates during active editing

    total_monthly_savings_goal = emergency_amt + secondary_monthly_target
    final_remaining_cash = disposable_before_savings - total_monthly_savings_goal

    return {
        "total_spent": total_spent,
        "disposable_before_savings": disposable_before_savings,
        "secondary_monthly_target": secondary_monthly_target,
        "months_remaining": months_to_secondary,
        "total_savings_goal": total_monthly_savings_goal,
        "final_remaining_cash": final_remaining_cash
    }

def add_manual_expense(ledger, amount, desc, category, merchant, date_str):
    if not amount or amount <= 0:
        return ledger, update_dataframe_view(ledger), "⚠️ Savings Buddy cannot log a $0 expense."
    if not desc:
        return ledger, update_dataframe_view(ledger), "⚠️ Please describe what this transaction was for!"
    try:
        tx_date = datetime.strptime(date_str, "%Y-%m-%d").date() if date_str else datetime.now().date()
    except ValueError:
        tx_date = datetime.now().date()

    record = {
        "Date": tx_date,
        "Amount": float(amount),
        "Description": desc,
        "Category": category,
        "Merchant": merchant if merchant else "General"
    }
    updated_ledger = ledger + [record]
    return updated_ledger, update_dataframe_view(updated_ledger), "✅ Expense successfully added to tracking stream!"

def upload_banking_csv(ledger, file_obj):
    if file_obj is None:
        return ledger, update_dataframe_view(ledger), "⚠️ No file detected."
    try:
        df = pd.read_csv(file_obj.name)
        df.columns = [col.strip() for col in df.columns]
        df['Category'] = df['Category'].astype(str).str.lower().str.strip()
        df['Amount'] = pd.to_numeric(df['Amount'], errors='coerce')
        df = df.dropna(subset=['Amount'])
        df = df[df['Amount'] > 0]

        if 'Date' not in df.columns:
            df['Date'] = datetime.now().date()
        else:
            df['Date'] = pd.to_datetime(df['Date'], errors='coerce').dt.date
            df = df.dropna(subset=['Date'])

        new_records = []
        for _, row in df.iterrows():
            cat_display = "Essential" if row['Category'] == 'essential' else "Non-Essential"
            new_records.append({
                "Date": row['Date'],
                "Amount": float(row['Amount']),
                "Description": row.get('Description', 'Imported Row'),
                "Category": cat_display,
                "Merchant": row.get('Merchant', 'General')
            })
        updated_ledger = ledger + new_records
        return updated_ledger, update_dataframe_view(updated_ledger), f"📈 Merged {len(new_records)} items from banking export successfully!"
    except Exception as e:
        return ledger, update_dataframe_view(ledger), f"⚠️ Spreadsheet parse error: {e}"

def update_dataframe_view(ledger):
    if not ledger:
        return pd.DataFrame(columns=["Date", "Amount", "Merchant", "Description", "Category"])
    return pd.DataFrame(ledger)

def compile_savings_report(ledger, name, currency, salary, emg_goal, has_sec, sec_name, sec_cost, sec_date):
    if not salary or salary <= 0:
        return "### ⚠️ Welcome! Please initialize your Salary and Savings Goals to compile the report."

    metrics = calculate_savings_metrics(ledger, salary, emg_goal, has_sec, sec_cost, sec_date)
    curr = currency if currency else "$"
    username = name if name else "Savings Companion"

    report = f"### 🦉 Okay {username}, Here is Your Savings Buddy Progress Report!\n\n"

    # Savings Goal Summaries
    report += f"#### 🎯 Active Savings Strategies\n"
    report += f"* **Compulsory Emergency Fund:** Allocating **{curr}{emg_goal:,.2f}** per month.\n"
    if has_sec and sec_name:
        report += f"* **Secondary Target Goal ('{sec_name}'):** Requires **{curr}{metrics['secondary_monthly_target']:,.2f}** monthly for the next **{metrics['months_remaining']} month(s)** to hit your {curr}{sec_cost:,.2f} baseline target.\n"
    report += f"* **Total Monthly Target Savings Volume:** **{curr}{metrics['total_savings_goal']:,.2f}**\n\n"

    # Cash Flow Breakdown Table
    report += f"#### 📊 Monthly Cash Flow Summary\n"
    report += f"| Income Base | Total Tracked Spending | Net Cash Before Savings | Final Leftover Balance |\n"
    report += f"| --- | --- | --- | --- |\n"
    report += f"| {curr}{salary:,.2f} | {curr}{metrics['total_spent']:,.2f} | {curr}{metrics['disposable_before_savings']:,.2f} | **{curr}{metrics['final_remaining_cash']:,.2f}** |\n\n"

    # Financial Guidance Warnings
    report += f"#### 🧠 Savings Feasibility Check\n"
    if metrics["final_remaining_cash"] >= 0:
        report += f"🏆 **Affordability Confirmed!** Your lifestyle spending leaves a healthy surplus. You can comfortably afford your target goals this month and keep an extra **{curr}{metrics['final_remaining_cash']:,.2f}** safety cushion raw liquid cash!"
    else:
        shortfall = abs(metrics["final_remaining_cash"])
        report += f"⚠️ **Savings Deficiency Warning.** Your combined savings targets exceed your current cash surplus by **{curr}{shortfall:,.2f}**. Consider filtering your Non-Essential expenses or extending your target secondary timeline to bring your targets back into equilibrium."

    return report

# =====================================================================
# HANDS-ON-AI INTEGRATION (Savings Context Router)
# =====================================================================
def savings_buddy_chat_response(user_message, chat_history, ledger, name, currency, salary, emg_goal, has_sec, sec_name, sec_cost, sec_date):
    if not user_message:
        return "", chat_history

    from hands_on_ai.chat import get_response

    username = name if name else "Friend"
    curr = currency if currency else "$"
    metrics = calculate_savings_metrics(ledger, salary, emg_goal, has_sec, sec_cost, sec_date)

    data_context = f"""
    Context User Name: {username}
    Preferred Currency: {curr}
    Monthly Salary: {curr}{salary:.2f}
    Tracked Outbound Expenses: {curr}{metrics['total_spent']:.2f}
    Compulsory Emergency Goal: {curr}{emg_goal:.2f}/month
    """
    if has_sec and sec_name:
        data_context += f"\nSecondary Goal: '{sec_name}' costing {curr}{sec_cost:.2f} (Target Monthly Contribution: {curr}{metrics['secondary_monthly_target']:.2f})"

    data_context += f"\nFinal Residual Cash Post-Savings Strategy: {curr}{metrics['final_remaining_cash']:.2f}"

    system_prompt = f"""You are 'Savings Buddy', a wise, encouraging, and warm AI financial counselor.
    Your job is to look over the user's explicit savings metrics and offer lifestyle adjustments, tactical saving tricks,
    and gentle advice on how they can balance their goals and live comfortably within their salary means.

    Active System Financial Parameters:
    {data_context}
    """

    try:
        ai_reply = get_response(prompt=user_message, system=system_prompt, personality="friendly")
        if "Connection error" in str(ai_reply) or "Error:" in str(ai_reply):
            raise ConnectionError()
    except Exception:
        ai_reply = f"☕ Hey {username}! The campus network is a little full right now, but mathematically speaking, you have {curr}{metrics['final_remaining_cash']:,.2f} remaining in your net tracking pocket. Ask me this question again in a tiny bit!"

    chat_history.append((user_message, ai_reply))
    return "", chat_history

# =====================================================================
# DYNAMIC VISUAL PALETTE ENGINE (Light, Dark, and Sepia UI Styles)
# =====================================================================
def apply_realtime_styles(theme_preset):
    """Dynamically rewrites browser UI components using standard responsive injection rules."""
    if theme_preset == "Dark Mode (Deep Onyx)":
        bg_color = "#121212"
        card_color = "#1e1e1e"
        btn_color = "#3a86ff"
        text_color = "#f5f5f5"
        banner_title = "#ffffff"
    elif theme_preset == "Sepia Mode (Vintage Linen)":
        bg_color = "#f4ecd8"
        card_color = "#e4d3b2"
        btn_color = "#8c6239"
        text_color = "#433422"
        banner_title = "#2b1d0c"
    else:
        # Default: Light Mode (Crisp Minimalist)
        bg_color = "#ffffff"
        card_color = "#f8f9fa"
        btn_color = "#007bff"
        text_color = "#212529"
        banner_title = "#000000"

    style_html = f"""
    <style>
        .gradio-container, body, html {{
            background-color: {bg_color} !important;
            color: {text_color} !important;
            font-family: 'Inter', sans-serif !important;
        }}
        .savings-banner {{
            background-color: {card_color} !important;
            border-radius: 12px;
            padding: 24px;
            text-align: center;
            border: 1px solid rgba(0,0,0,0.05);
        }}
        .savings-banner h1 {{ color: {banner_title} !important; margin: 0; }}
        .savings-banner p {{ color: {text_color} !important; margin: 6px 0 0 0; opacity: 0.8; }}

        button.primary {{
            background-color: {btn_color} !important;
            color: white !important;
            border: none !important;
            font-weight: bold !important;
            border-radius: 8px !important;
        }}
        button.primary:hover {{ opacity: 0.9 !important; }}
    </style>
    """
    return style_html, f"🎨 Theme updated seamlessly to '{theme_preset}'!"

# =====================================================================
# GRADIO APPLICATION INTERFACE CONTEXT
# =====================================================================
with gr.Blocks(title="Savings Buddy Framework") as app:
    state_ledger = gr.State([])
    style_injection_box = gr.HTML("")

    # Visual Theme Controls Panel
    with gr.Row():
        gr.Markdown("## ☕ Personalize Your Savings Dashboard Environment")
    with gr.Row():
        theme_selector = gr.Radio(
            choices=["Light Mode (Crisp Minimalist)", "Dark Mode (Deep Onyx)", "Sepia Mode (Vintage Linen)"],
            value="Light Mode (Crisp Minimalist)", label="Select App Display Aesthetic"
        )
        apply_theme_btn = gr.Button("🎨 Re-render Display Theme", variant="secondary")

    gr.Markdown("---")

    # App Branding Header
    with gr.Row():
        gr.HTML("""
        <div class="savings-banner">
            <h1>🐷 Your Savings Buddy Tracker</h1>
            <p>Let's map out your earnings, insulate your emergency funds, and calculate realistic paths to your dream milestones! 🌟</p>
        </div>
        """)

    # STEP 1: Core Profile Initializer
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🌟 Step 1: Initialize Your Core Demographics & Targets")
            user_name = gr.Textbox(placeholder="What name do you prefer?", label="First Name")
            user_curr = gr.Textbox(value="$", placeholder="e.g. $, €, £, Pula", label="Preferred Currency Symbol")
            user_salary = gr.Number(value=3000, label="💰 Monthly Net Salary Income")

            gr.Markdown("---")
            gr.Markdown("#### 🛡️ Emergency Fund Configuration (Compulsory Baseline)")
            emg_target = gr.Number(value=300, label="Monthly Emergency Fund Retainer Target", info="The set amount you promise to isolate every single month.")

        with gr.Column():
            gr.Markdown("#### ✈️ Custom Secondary Savings Goal Horizon")
            has_secondary = gr.Checkbox(label="Activate a Secondary Custom Goal Tracker?", value=False)
            sec_name = gr.Textbox(placeholder="e.g. Holiday in Botswana, New Limousine", label="Secondary Goal Name")
            sec_cost = gr.Number(value=5000, label="Total Target Cost Estimate")
            sec_date = gr.Textbox(value="2027-06-12", placeholder="YYYY-MM-DD", label="Target Achievement Date Cutoff")

    # STEP 2: Spending Leak Ingestion Channel
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 📊 Step 2: Feed in Current Month Expenses (To Deduct Savings Feasibility)")
            with gr.Tabs():
                with gr.TabItem("📝 Manual Cost Logging"):
                    with gr.Row():
                        tx_amt = gr.Number(label="Expense Amount Cost")
                        tx_desc = gr.Textbox(label="Transaction Description", placeholder="Groceries, Cake, Uber, etc.")
                    with gr.Row():
                        tx_cat = gr.Dropdown(choices=["Essential", "Non-Essential"], value="Essential", label="Expense Category Sorting")
                        tx_merch = gr.Textbox(label="Merchant Name Source")
                    tx_date = gr.Textbox(label="Transaction Date", value=datetime.now().strftime("%Y-%m-%d"))
                    manual_btn = gr.Button("➕ Log Transaction to Memory", variant="primary")

                with gr.TabItem("📎 Bank Statement CSV Upload"):
                    file_upload = gr.File(file_types=[".csv"], label="Drag & Drop Statement File")
                    csv_btn = gr.Button("🚀 Bulk Parse and Append Rows", variant="primary")

    # Live Data Verification Terminal
    with gr.Row():
        with gr.Column():
            status_ticker = gr.Textbox(value="📭 Ledger clean. Awaiting entry nodes.", label="System Status Terminal Updates", interactive=False)
            ledger_view = gr.DataFrame(datatype=["str", "number", "str", "str", "str"], value=pd.DataFrame(columns=["Date", "Amount", "Merchant", "Description", "Category"]), interactive=False)
            clear_btn = gr.Button("🗑️ Clear Expense Ledger Assets / Wipe Fresh", variant="stop")

    # STEPS 3 & 4: Strategic Forecast Report + Live Interactive Savings Advisor
    with gr.Row():
        with gr.Column():
            gr.Markdown("### 🦉 Step 3: Compile Forecasting Financial Report Statements")
            finalize_btn = gr.Button("📊 Process Feasibility Analytics Report", variant="primary")
            report_box = gr.Markdown("✨ *Your custom calculations framework statements will populate here...*")

        with gr.Column():
            gr.Markdown("### 💬 Step 4: Consult Your Dedicated AI Savings Companion")
            chatbot_ui = gr.Chatbot(label="Savings Strategic Consultation Box")
            user_msg = gr.Textbox(placeholder="Ask Advisor: 'Can I afford my Holiday in Botswana if I log an extra $200 expense?'", label="Type Chat Message...")
            chat_send_btn = gr.Button("💬 Dispatch Query", variant="primary")

    # =====================================================================
    # COMPONENT WIRE FUNCTION INTERACTION MAPS
    # =====================================================================

    # Custom CSS Theme Swapping Router Chain
    apply_theme_btn.click(fn=apply_realtime_styles, inputs=[theme_selector], outputs=[style_injection_box, status_ticker])

    # Logging Ingest Hooks
    manual_btn.click(
        fn=add_manual_expense, inputs=[state_ledger, tx_amt, tx_desc, tx_cat, tx_merch, tx_date], outputs=[state_ledger, ledger_view, status_ticker]
    ).then(fn=lambda: (None, "", "Essential", ""), outputs=[tx_amt, tx_desc, tx_cat, tx_merch])

    csv_btn.click(fn=upload_banking_csv, inputs=[state_ledger, file_upload], outputs=[state_ledger, ledger_view, status_ticker])
    clear_btn.click(fn=lambda: ([], update_dataframe_view([]), "🗑️ Expense tracking array flushed completely."), outputs=[state_ledger, ledger_view, status_ticker])

    # Compiled Analytic Reporting Hook
    finalize_btn.click(
        fn=compile_savings_report,
        inputs=[state_ledger, user_name, user_curr, user_salary, emg_target, has_secondary, sec_name, sec_cost, sec_date],
        outputs=[report_box]
    )

    # Chatbot Evaluation Dispatch Loop
    chat_send_btn.click(
        fn=savings_buddy_chat_response,
        inputs=[user_msg, chatbot_ui, state_ledger, user_name, user_curr, user_salary, emg_target, has_secondary, sec_name, sec_cost, sec_date],
        outputs=[user_msg, chatbot_ui]
    )

app.launch(debug=True)

/tmp/ipykernel_6434/1354580892.py:317: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot_ui = gr.Chatbot(label="Savings Strategic Consultation Box")
/tmp/ipykernel_6434/1354580892.py:317: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot_ui = gr.Chatbot(label="Savings Strategic Consultation Box")


It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://b9b52e474b8633445b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
